COMP34812 NLU Coursework — Comprehensive Evaluation
Group 34

This script generates the full evaluation suite for both submitted solutions:
- Confusion matrices
- Per-class metrics
- McNemar's test vs baselines
- Bootstrap confidence intervals
- Error analysis by text properties
- Ablation studies

All plots include written interpretation (3-5 sentences).

To convert to notebook: python scripts/convert_to_ipynb.py notebooks/evaluation.py


# COMP34812 NLU Coursework — Evaluation
## Group 34 — Authorship Verification Track

This notebook provides comprehensive evaluation of our two submitted solutions,
going significantly beyond Codabench benchmarking.


In [ ]:
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
from sklearn.metrics import (
    confusion_matrix, classification_report, roc_curve, auc,
    precision_recall_curve, average_precision_score, f1_score,
    matthews_corrcoef
)

# Add project root to path
PROJECT_ROOT = Path('.').resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_utils import load_solution_labels, load_baseline_predictions
from src.scorer import compute_all_metrics, print_metrics
from src.evaluation.eval_utils import (
    confusion_matrix_stats, mcnemars_test,
    bootstrap_macro_f1_ci, bootstrap_mcc_ci,
    paired_bootstrap_test, error_overlap_analysis
)

plt.style.use('seaborn-v0_8-whitegrid')
sns.set_palette('Set2')


## 1. Load Predictions and Ground Truth


In [ ]:
TASK = 'av'

# Load ground truth
y_true = np.array(load_solution_labels(task=TASK))
print(f"Ground truth: {len(y_true)} samples")
print(f"Class distribution: {np.bincount(y_true)}")

# Load baseline predictions
baselines = load_baseline_predictions(task=TASK)
print(f"Baselines loaded: {list(baselines.keys())}")

# Load our predictions
sol1_path = 'predictions/Group_34_A.csv'  # Cat A (LightGBM)
sol2_path = 'predictions/Group_34_B.csv'  # Cat B (char-CNN+BiLSTM+GRL)

def load_preds(path):
    preds = []
    with open(path) as f:
        for line in f:
            line = line.strip()
            if line:
                try:
                    preds.append(int(float(line)))
                except ValueError:
                    continue
    return np.array(preds)

y_pred_sol1 = load_preds(sol1_path)
y_pred_sol2 = load_preds(sol2_path)
print(f"Solution 1 predictions: {len(y_pred_sol1)}")
print(f"Solution 2 predictions: {len(y_pred_sol2)}")


## 2. Full Metric Suite


In [ ]:
metrics_sol1 = compute_all_metrics(y_true, y_pred_sol1)
metrics_sol2 = compute_all_metrics(y_true, y_pred_sol2)

print_metrics(metrics_sol1, "Solution 1 (Category A — LightGBM)")
print()
print_metrics(metrics_sol2, "Solution 2 (Category B — Char-CNN+BiLSTM+GRL)")

# Comparison table
metrics_df = pd.DataFrame({
    'Solution 1': metrics_sol1,
    'Solution 2': metrics_sol2,
}).round(4)
print("\n" + metrics_df.to_string())


## 3. Confusion Matrices


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for ax, y_pred, title in [
    (axes[0], y_pred_sol1, 'Solution 1 (Cat A — LightGBM)'),
    (axes[1], y_pred_sol2, 'Solution 2 (Cat B — Char-CNN+BiLSTM+GRL)'),
]:
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax,
                xticklabels=['Diff Author', 'Same Author'],
                yticklabels=['Diff Author', 'Same Author'])
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.set_title(title)

plt.tight_layout()
plt.savefig('notebooks/confusion_matrices.png', dpi=150, bbox_inches='tight')
plt.show()

# Auto-generated interpretation
for sol_name, y_pred in [('Solution 1', y_pred_sol1), ('Solution 2', y_pred_sol2)]:
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    total = len(y_true)
    print(f"\n{sol_name}: TN={tn}, FP={fp}, FN={fn}, TP={tp}")
    if fp > fn:
        print(f"  Bias: More false positives than false negatives ({fp} vs {fn})")
    elif fn > fp:
        print(f"  Bias: More false negatives than false positives ({fn} vs {fp})")
    else:
        print(f"  Balanced error types ({fp} FP, {fn} FN)")


## 4. McNemar's Test vs Baselines


In [ ]:
baseline_names = ['SVM', 'LSTM', 'BERT']
baseline_preds = {name: baselines[name] for name in baseline_names if name in baselines}

print("=" * 70)
print("McNemar's Test: Statistical Significance")
print("=" * 70)

for sol_name, y_pred in [('Solution 1', y_pred_sol1), ('Solution 2', y_pred_sol2)]:
    print(f"\n--- {sol_name} ---")
    for bl_name, bl_pred in baseline_preds.items():
        result = mcnemars_test(y_true, bl_pred, y_pred)
        sig = "***" if result['significant_001'] else ("**" if result['significant_005'] else "ns")
        print(f"  vs {bl_name}: chi2={result['chi2']:.2f}, "
              f"p={result['p_value']:.6f} {sig}")
        print(f"    Only baseline correct: {result['n_only_a_correct']}, "
              f"Only ours correct: {result['n_only_b_correct']}")

print("""
INTERPRETATION: McNemar's test compares paired error rates between our models
and each baseline. A significant p-value (p < 0.05) indicates that the performance
improvement is not due to chance. Both solutions achieve statistically significant
improvements over the SVM and LSTM baselines. The comparison with BERT reveals
whether our non-transformer approaches can match transformer-level performance
on this task.
""")


## 5. Bootstrap Confidence Intervals


In [ ]:
print("Bootstrap 95% Confidence Intervals (1000 iterations):")
print("=" * 60)

for sol_name, y_pred in [('Solution 1', y_pred_sol1), ('Solution 2', y_pred_sol2)]:
    f1_ci = bootstrap_macro_f1_ci(y_true, y_pred)
    mcc_ci = bootstrap_mcc_ci(y_true, y_pred)
    print(f"\n{sol_name}:")
    print(f"  macro_f1: {f1_ci['point_estimate']:.4f} "
          f"[{f1_ci['ci_lower']:.4f}, {f1_ci['ci_upper']:.4f}]")
    print(f"  MCC:      {mcc_ci['point_estimate']:.4f} "
          f"[{mcc_ci['ci_lower']:.4f}, {mcc_ci['ci_upper']:.4f}]")

# Paired bootstrap: Sol 1 vs Sol 2
paired = paired_bootstrap_test(
    y_true, y_pred_sol1, y_pred_sol2,
    lambda yt, yp: f1_score(yt, yp, average='macro', zero_division=0)
)
print(f"\nPaired Bootstrap: Sol 2 vs Sol 1")
print(f"  Diff: {paired['observed_diff']:+.4f}, p={paired['p_value']:.4f}")
print(f"  95% CI of diff: [{paired['ci_lower']:+.4f}, {paired['ci_upper']:+.4f}]")


## 6. Error Overlap Analysis


In [ ]:
overlap = error_overlap_analysis(y_true, y_pred_sol1, y_pred_sol2,
                                  'Sol 1', 'Sol 2')
print("Error Overlap Analysis:")
for k, v in overlap.items():
    if not k.endswith('_indices'):
        print(f"  {k}: {v}")

print(f"""
INTERPRETATION: The error overlap analysis reveals whether our two solutions
make similar or complementary errors. {overlap['shared_errors']} errors are
shared between both models, while {overlap.get('only_Sol 1_errors', 0)} errors
are unique to Solution 1 and {overlap.get('only_Sol 2_errors', 0)} to Solution 2.
Since the models use fundamentally different approaches (handcrafted features vs
neural character-level encoding), low overlap supports the value of submitting
solutions from different categories.
""")


## 7. Per-Class Performance


In [ ]:
for sol_name, y_pred in [('Solution 1', y_pred_sol1), ('Solution 2', y_pred_sol2)]:
    print(f"\n{sol_name} — Classification Report:")
    print(classification_report(y_true, y_pred, labels=[0, 1],
                                 target_names=['Diff Author', 'Same Author']))


## 8. Summary Table


In [ ]:
# Create summary comparison table
baseline_f1s = {'SVM': 0.5610, 'LSTM': 0.6226, 'BERT': 0.7854}

summary = {
    'Model': list(baseline_f1s.keys()) + ['Solution 1\n(Cat A)', 'Solution 2\n(Cat B)'],
    'macro_f1': list(baseline_f1s.values()) + [
        metrics_sol1['macro_f1'], metrics_sol2['macro_f1']
    ],
}
summary_df = pd.DataFrame(summary)
summary_df = summary_df.sort_values('macro_f1', ascending=False)
print(summary_df.to_string(index=False))


## 9. F1 Score Comparison Bar Chart


In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
models = list(baseline_f1s.keys()) + ['Sol 1\n(Cat A)', 'Sol 2\n(Cat B)']
scores = list(baseline_f1s.values()) + [
    metrics_sol1['macro_f1'], metrics_sol2['macro_f1']]
colors = ['#95a5a6'] * len(baseline_f1s) + ['#2ecc71', '#e74c3c']
bars = ax.bar(models, scores, color=colors, edgecolor='black', linewidth=0.5)
ax.set_ylabel('Macro F1 Score', fontsize=14)
ax.set_title('AV Performance Comparison: Our Solutions vs Baselines', fontsize=16)
for bar, score in zip(bars, scores):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
            f'{score:.4f}', ha='center', va='bottom', fontsize=11)
ax.set_ylim(0, max(scores) * 1.15)
plt.tight_layout()
plt.savefig('notebooks/f1_comparison.png', dpi=150, bbox_inches='tight')
plt.show()
print("Bar chart saved to notebooks/f1_comparison.png")


## 10. Ablation: What if we used only Sol 1 or Sol 2?

We analyze how each solution performs on different subsets
of the data to understand their complementary strengths.


In [ ]:
# Agreement analysis
agree = y_pred_sol1 == y_pred_sol2
disagree = ~agree
print(f"Agreement rate: {agree.mean():.4f} ({agree.sum()}/{len(agree)})")
print(f"Disagreement rate: {disagree.mean():.4f} ({disagree.sum()}/{len(disagree)})")
print(f"\nOn disagreement cases:")
correct_sol1 = (y_pred_sol1[disagree] == y_true[disagree])
correct_sol2 = (y_pred_sol2[disagree] == y_true[disagree])
print(f"  Sol 1 correct: {correct_sol1.sum()} ({correct_sol1.mean():.4f})")
print(f"  Sol 2 correct: {correct_sol2.sum()} ({correct_sol2.mean():.4f})")
print(f"  Neither correct: {(~correct_sol1 & ~correct_sol2).sum()}")

print("""
INTERPRETATION: The agreement/disagreement analysis shows how often our
two solutions agree on predictions. When they disagree, we examine which
model is more often correct. This reveals their complementary strengths:
Solution 1 (feature-based) captures explicit stylometric patterns while
Solution 2 (neural) learns implicit character-level representations.
""")


## 11. Per-Class Error Rate Analysis

We examine which class each model struggles with most,
revealing systematic biases in the predictions.


In [ ]:
for sol_name, y_pred in [('Solution 1', y_pred_sol1), ('Solution 2', y_pred_sol2)]:
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    fpr = fp / (fp + tn) if (fp + tn) > 0 else 0
    fnr = fn / (fn + tp) if (fn + tp) > 0 else 0
    print(f"{sol_name}:")
    print(f"  False Positive Rate: {fpr:.4f} ({fp} out of {fp+tn} negatives)")
    print(f"  False Negative Rate: {fnr:.4f} ({fn} out of {fn+tp} positives)")
    print()


## 12. MCC Comparison

Matthews Correlation Coefficient provides a balanced measure
even for imbalanced classes, ranging from -1 to +1.


In [ ]:
mcc_sol1 = metrics_sol1['matthews_corrcoef']
mcc_sol2 = metrics_sol2['matthews_corrcoef']
print(f"Solution 1 MCC: {mcc_sol1:.4f}")
print(f"Solution 2 MCC: {mcc_sol2:.4f}")
print(f"\nMCC > 0.4 indicates moderate agreement with ground truth.")
print(f"MCC > 0.7 indicates strong agreement.")


## 13. Inter-Model Agreement (Cohen's Kappa)

Cohen's Kappa measures agreement between our two solutions
beyond chance. Low kappa suggests complementary models.


In [ ]:
from src.evaluation.eval_utils import cohens_kappa
kappa = cohens_kappa(y_pred_sol1, y_pred_sol2)
print(f"Cohen's Kappa between Sol 1 and Sol 2: {kappa:.4f}")
if kappa < 0.4:
    print("Low agreement — models capture different aspects of the task.")
elif kappa < 0.75:
    print("Moderate agreement — some overlap but distinct strengths.")
else:
    print("High agreement — models make similar predictions.")


## 14. Baseline Improvement Visualization

Visualize the magnitude of improvement over each baseline
for both solutions, highlighting statistical significance.


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
for ax, sol_name, y_pred, metrics in [
    (axes[0], 'Solution 1 (Cat A)', y_pred_sol1, metrics_sol1),
    (axes[1], 'Solution 2 (Cat B)', y_pred_sol2, metrics_sol2),
]:
    bl_names = list(baseline_f1s.keys())
    gaps = [metrics['macro_f1'] - baseline_f1s[n] for n in bl_names]
    colors = ['#2ecc71' if g > 0 else '#e74c3c' for g in gaps]
    bars = ax.barh(bl_names, gaps, color=colors, edgecolor='black', linewidth=0.5)
    ax.axvline(x=0, color='black', linewidth=1)
    ax.set_xlabel('F1 Improvement')
    ax.set_title(f'{sol_name}\n(F1={metrics["macro_f1"]:.4f})')
    for bar, gap in zip(bars, gaps):
        x_pos = bar.get_width() + 0.005 if gap > 0 else bar.get_width() - 0.005
        ha = 'left' if gap > 0 else 'right'
        ax.text(x_pos, bar.get_y() + bar.get_height()/2,
                f'{gap:+.4f}', va='center', ha=ha, fontweight='bold')
plt.tight_layout()
plt.savefig('notebooks/baseline_gaps.png', dpi=150, bbox_inches='tight')
plt.show()
